# Vector Index 생성하기

PDF 지식그래프의 **TextElement** 노드에 임베딩을 추가하고 벡터 인덱스를 생성합니다.

## Neo4j GenAI Plugin이란?

Neo4j GenAI Plugin을 사용하면 **Neo4j 데이터베이스 내부에서 직접 임베딩을 생성**할 수 있습니다.

- `ai.text.embed()`: 단일 텍스트 임베딩
- `ai.text.embedBatch()`: 배치 임베딩

참고: https://neo4j.com/docs/genai/plugin/current/embeddings/

## 1. 환경 설정 및 연결

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_USERNAME")  # 데이터베이스명
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print(f"Neo4j URI: {NEO4J_URI}")

Neo4j URI: neo4j://127.0.0.1:7687


In [2]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

driver.verify_connectivity()

## 2. 현재 그래프 상태 확인

PDF 지식그래프의 TextElement 노드가 얼마나 있는지 확인합니다.

In [ ]:
# TextElement 노드 개수
result = driver.execute_query("""
    MATCH (t:TextElement)
    RETURN count(t) AS total
""", database_=NEO4J_DATABASE)
total = result.records[0]["total"]
print(f"TextElement 노드 총 개수: {total}")

# 샘플 TextElement 확인
result = driver.execute_query("""
    MATCH (t:TextElement)
    RETURN t.element_id AS id,
           substring(t.content, 0, 100) AS content_preview,
           t.page AS page,
           t.embedding IS NOT NULL AS has_embedding
    LIMIT 3
""", database_=NEO4J_DATABASE)

print("\n샘플 TextElement 노드:")
for record in result.records:
    print(f"\n  ID: {record['id']}")
    print(f"  Page: {record['page']}")
    print(f"  Content: {record['content_preview']}...")
    print(f"  Has Embedding: {record['has_embedding']}")

TextElement 노드 총 개수: 312


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: embedding)} {position: line: 6, column: 14, offset: 158} for query: '\n    MATCH (t:TextElement)\n    RETURN t.element_id AS id,\n           substring(t.content, 0, 100) AS content_preview,\n           t.page AS page,\n           t.embedding IS NOT NULL AS has_embedding\n    LIMIT 3\n'



샘플 TextElement 노드:

  ID: text_0000
  Page: 4
  Content: 1/5 Nvidia Nemotron3·Cosmos Reason2 (LLM) 1/11 K-Exaone (LLM, 한국)...
  Has Embedding: False

  ID: text_0001
  Page: 4
  Content: 1/13 Google Veo 3.1(영상) 1/26 Qwen-Max-Thinking (LLM),...
  Has Embedding: False

  ID: text_0002
  Page: 4
  Content: 1/27 Kimi 2.5 (LLM) 2/5 Claude Opus 4.6, GPT-5.3 Codex...
  Has Embedding: False


## 3. ai.text.embed() 테스트

먼저 단일 텍스트에 대해 `ai.text.embed()` 함수가 작동하는지 확인합니다.

```SQL
RETURN ai.text.embed(
  "This is a test sentence.",
  "openai",
  {
    token: "sk-proj-...",
    model: "text-embedding-3-small"
  }
) AS embedding;
```


### 단일 노드에 임베딩 저장하기

`ai.text.embed()`를 사용하여 하나의 TextElement 노드에 임베딩을 추가하는 예제입니다.

In [ ]:
# 임베딩이 없는 첫 번째 TextElement 노드에 임베딩 추가
result = driver.execute_query("""
    MATCH (t:TextElement)
    WHERE t.embedding IS NULL AND t.content IS NOT NULL
    WITH t
    LIMIT 1

    WITH t, ai.text.embed(
        t.content,
        'openai',
        {
            token: $openai_token,
            model: 'text-embedding-3-small'
        }
    ) AS vector

    SET t.embedding = vector

    RETURN
        t.element_id AS element_id,
        t.page AS page,
        size(t.embedding) AS embedding_dim,
        substring(t.content, 0, 100) AS content_preview
""", openai_token=OPENAI_API_KEY, database_=NEO4J_DATABASE)

record = result.records[0]
print(f"단일 노드에 임베딩 추가 성공!")
print(f"   Element ID: {record['element_id']}")
print(f"   Page: {record['page']}")
print(f"   임베딩 차원: {record['embedding_dim']}")
print(f"   내용: {record['content_preview']}...")

단일 노드에 임베딩 추가 성공!
   Element ID: text_0000
   Page: 4
   임베딩 차원: 1536
   내용: 1/5 Nvidia Nemotron3·Cosmos Reason2 (LLM) 1/11 K-Exaone (LLM, 한국)...


## 4. ai.text.embedBatch()로 임베딩 추가 (소량 테스트)

In [ ]:
TEST_LIMIT = 10

result = driver.execute_query("""
    // 1. 임베딩이 없는 TextElement 노드 가져오기
    MATCH (t:TextElement)
    WHERE t.embedding IS NULL
    WITH t
    LIMIT $test_limit

    // 2. content를 문자열 리스트로 추출
    WITH collect(t) AS nodesList
    WITH nodesList,
        [node IN nodesList | node.content] AS contentBatch

    // 3. ai.text.embedBatch() 호출
    CALL ai.text.embedBatch(
        contentBatch,
        'openai',
        {
            token: $openai_token,
            model: 'text-embedding-3-small'
        }
    ) YIELD index, vector

    // 4. 해당 노드에 임베딩 저장
    WITH nodesList, index, vector
    WITH nodesList[index] AS node, vector
    SET node.embedding = vector

    RETURN count(*) AS updated
""", openai_token=OPENAI_API_KEY, test_limit=TEST_LIMIT, database_=NEO4J_DATABASE)

updated = result.records[0]["updated"]
print(f"{updated}개 노드에 임베딩 추가 완료!")

10개 노드에 임베딩 추가 완료!


## 5. 임베딩 추가 확인

In [ ]:
result = driver.execute_query("""
    MATCH (t:TextElement)
    RETURN
        count(t) AS total,
        sum(CASE WHEN t.embedding IS NOT NULL THEN 1 ELSE 0 END) AS with_embedding,
        sum(CASE WHEN t.embedding IS NULL THEN 1 ELSE 0 END) AS without_embedding
""", database_=NEO4J_DATABASE)

record = result.records[0]
total = record['total']
with_emb = record['with_embedding']
without_emb = record['without_embedding']

print(f"TextElement 노드 통계:")
print(f"  - 전체: {total}")
print(f"  - 임베딩 있음: {with_emb} ({with_emb/total*100:.1f}%)")
print(f"  - 임베딩 없음: {without_emb}")

TextElement 노드 통계:
  - 전체: 312
  - 임베딩 있음: 11 (3.5%)
  - 임베딩 없음: 301


## 6. 전체 노드에 임베딩 추가 (배치 처리)

모든 TextElement 노드에 임베딩을 추가합니다.
- 100개씩 배치로 처리

In [ ]:
BATCH_SIZE = 100  # 배치 크기

total_updated = 0

while True:
    result = driver.execute_query("""
        MATCH (t:TextElement)
        WHERE t.embedding IS NULL
        WITH t
        LIMIT $batch_size

        WITH collect(t) AS batch
        WITH batch,
             [node IN batch | node.content] AS contentBatch

        CALL ai.text.embedBatch(
            contentBatch,
            'openai',
            {
                token: $openai_token,
                model: 'text-embedding-3-small'
            }
        ) YIELD index, vector

        WITH batch[index] AS node, vector
        SET node.embedding = vector

        RETURN count(*) AS updated
    """, openai_token=OPENAI_API_KEY, batch_size=BATCH_SIZE, database_=NEO4J_DATABASE)

    updated = result.records[0]["updated"]

    if updated == 0:
        break

    total_updated += updated
    print(f"{updated}개 처리, 누적 {total_updated}")

print(f"\n전체 임베딩 추가 완료: {total_updated}개 노드")

100개 처리, 누적 100
100개 처리, 누적 200
100개 처리, 누적 300
1개 처리, 누적 301

전체 임베딩 추가 완료: 301개 노드


## 7. 벡터 인덱스 생성

Neo4j의 벡터 인덱스를 생성하여 빠른 유사도 검색을 가능하게 합니다.

1. **Cypher로 직접 생성**
2. **neo4j-graphrag 패키지 사용**

**두 방법의 차이점:**

| 방법 | 장점 | 단점 |
|------|------|------|
| **Cypher 직접 사용** | - 세밀한 제어 가능<br>- IF NOT EXISTS 조건 지원<br>- Neo4j 표준 방식 | - Cypher 문법 필요 |
| **neo4j-graphrag 패키지** | - Python 코드로 간결<br>- 타입 안정성<br>- IDE 자동완성 | - 예외 처리 필요<br>- 추가 패키지 의존성 |


In [ ]:
INDEX_NAME = "textElementEmbedding"
VECTOR_DIM = 1536  # text-embedding-3-small의 차원

# 기존 인덱스 확인
result = driver.execute_query("""
    SHOW INDEXES
    YIELD name, type
    WHERE name = $index_name AND type = 'VECTOR'
    RETURN count(*) AS count
""", index_name=INDEX_NAME, database_=NEO4J_DATABASE)

exists = result.records[0]['count'] > 0

if exists:
    print(f"벡터 인덱스 '{INDEX_NAME}'이 이미 존재합니다.")
else:
    # 벡터 인덱스 생성
    driver.execute_query(f"""
        CREATE VECTOR INDEX {INDEX_NAME} IF NOT EXISTS
        FOR (t:TextElement)
        ON t.embedding
        OPTIONS {{
          indexConfig: {{
            `vector.dimensions`: $vector_dim,
            `vector.similarity_function`: 'cosine'
          }}
        }}
    """, vector_dim=VECTOR_DIM, database_=NEO4J_DATABASE)

    print(f"벡터 인덱스 '{INDEX_NAME}' 생성 완료!")
    print(f"   - 차원: {VECTOR_DIM}")
    print(f"   - 유사도 함수: cosine")

벡터 인덱스 'textElementEmbedding' 생성 완료!
   - 차원: 1536
   - 유사도 함수: cosine


### 방법 2: neo4j-graphrag 패키지 사용

`neo4j-graphrag` 패키지의 [`create_vector_index`](https://neo4j.com/docs/neo4j-graphrag-python/current/user_guide_rag.html#create-a-vector-index) 함수를 사용하여 Python에서 벡터 인덱스를 생성할 수도 있습니다.

In [9]:
from neo4j_graphrag.indexes import create_vector_index

INDEX_NAME_ALT = "textElementEmbedding_v2"
VECTOR_DIM = 1536

try:
    create_vector_index(
        driver,
        INDEX_NAME_ALT,
        label="TextElement",
        embedding_property="embedding",
        dimensions=VECTOR_DIM,
        similarity_fn="cosine",  # 벡터 간 유사도 계산 방식
    )

    print(f"벡터 인덱스 '{INDEX_NAME_ALT}' 생성 완료!")
    print(f"   - 레이블: TextElement")
    print(f"   - 속성: embedding")
    print(f"   - 차원: {VECTOR_DIM}")
    print(f"   - 유사도 함수: cosine")

except Exception as e:
    print(f"오류 발생: {e}")

벡터 인덱스 'textElementEmbedding_v2' 생성 완료!
   - 레이블: TextElement
   - 속성: embedding
   - 차원: 1536
   - 유사도 함수: cosine


## 8. 인덱스 상태 확인

In [ ]:
result = driver.execute_query("""
    SHOW INDEXES
    YIELD name, type, labelsOrTypes, properties, state
    WHERE type = 'VECTOR'
    RETURN name, labelsOrTypes, properties, state
""", database_=NEO4J_DATABASE)

print("벡터 인덱스 목록:")
for record in result.records:
    print(f"\n  이름: {record['name']}")
    print(f"  레이블: {record['labelsOrTypes']}")
    print(f"  속성: {record['properties']}")
    print(f"  상태: {record['state']}")

벡터 인덱스 목록:

  이름: textElementEmbedding
  레이블: ['TextElement']
  속성: ['embedding']
  상태: ONLINE


## 9. 벡터 검색 테스트

생성한 벡터 인덱스를 사용하여 유사도 검색을 테스트합니다.
`ai.text.embed()`로 쿼리 임베딩을 생성한 후 `db.index.vector.queryNodes()`로 검색합니다.

In [ ]:
query_text = "AI와 관련된 내용"  # 원하는 검색어로 변경하세요
top_k = 3

result = driver.execute_query("""
    // 1. 쿼리 텍스트를 임베딩으로 변환
    WITH ai.text.embed(
        $query_text,
        'openai',
        {
            token: $openai_token,
            model: 'text-embedding-3-small'
        }
    ) AS query_vector

    // 2. 벡터 검색
    CALL db.index.vector.queryNodes(
        'textElementEmbedding',
        $top_k,
        query_vector
    )
    YIELD node, score

    RETURN
        node.element_id AS element_id,
        node.content AS content,
        node.page AS page,
        score
    ORDER BY score DESC
""", query_text=query_text, openai_token=OPENAI_API_KEY, top_k=top_k, database_=NEO4J_DATABASE)

print(f"🔍 검색 쿼리: '{query_text}'")
print(f"\n📊 상위 {top_k}개 결과:\n")

for i, record in enumerate(result.records, 1):
    print(f"[{i}] 유사도: {record['score']:.4f}")
    print(f"    페이지: {record['page']}")
    print(f"    내용: {record['content'][:150]}...")
    print()

🔍 검색 쿼리: 'AI와 관련된 내용'

📊 상위 3개 결과:

[1] 유사도: 0.7763
    페이지: 10
    내용: Ÿ AI 저작권 가이드라인 및 관련 규정을 개선하고 , A AI 연구 및 응용 프로그램 개발을 지원하기 위해 정부와 민간 데이터의 개방을 적극적으로 추진...

[2] 유사도: 0.7585
    페이지: 10
    내용: Ÿ 민관 협력으로 미래 지향적 관점에서 대규모 AI 컴퓨팅 역량을 구축하여 , A AI R&D 및 응용 분야 발전 지원...

[3] 유사도: 0.7546
    페이지: 27
    내용: n 생성형 AI의 위험에 대응해 기술기업과 정부 , 학교 , 교사 , 부모 , 학생 등 모든 이해관계자가 적극적으로 관리에 나서야 하며 , 해결 방안과 관리 체계로 다음 3가지 방향과 권고안을 제시...



In [12]:
driver.close()